# Project Update 1 — Data Engineering for Streaming Code-Switching

Pipeline: **SwitchLingua text → word-level LID → predictive labels (y_sw, y_dur) → sub-word alignment → Streaming DataLoader** yielding context windows `[x_{t-N}, …, x_t]`.

Label rules (from the project spec):
- `y_sw(t) = 1` if `LID(x_{t+1}) ≠ LID(x_t)`, else `0`. The last token has no next token → ignored (`-100`).
- `y_dur(t)` = length bin of the new-language segment starting at `x_{t+1}` when `y_sw(t)=1`: **0** = 1–2 tokens, **1** = 3–6, **2** = 7+. Otherwise `-1` (ignored).

In [2]:
# !pip install datasets transformers torch pandas wordfreq
import torch, pandas as pd
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader, get_worker_info
from wordfreq import zipf_frequency

## 1. Load SwitchLingua and inspect the schema
The dataset is gated: accept its conditions on the Hugging Face page and log in first (`huggingface-cli login` or `from huggingface_hub import login; login()`).

In [23]:
ds = load_dataset("Shelton1013/SwitchLingua_text")
ds["train"]

Dataset({
    features: ['topic', 'tense', 'perspective', 'cs_ratio', 'gender', 'age', 'education_level', 'first_language', 'second_language', 'conversation_type', 'cs_function', 'cs_type', 'data_generation_result', 'fluency_result', 'naturalness_result', 'cs_ratio_result', 'social_cultural_result', 'summary', 'score'],
    num_rows: 234172
})

## 2. Configuration
The dataset has **no token-level language labels**: each row gives `first_language`, `second_language` and the sentence in `data_generation_result`. Word-level LIDs are therefore derived in Section 3.

`SELECTED_PAIRS` should match the language pairs chosen in Part 1.
Languages written in a different script from English (Arabic, Hindi, Cantonese, …) can be labeled by script.
Latin-script languages (Spanish, German) are labeled with the frequency-based LID in Section 3.

In [24]:
import ast

TEXT_COL = "data_generation_result"
# Six typologically diverse pairs:
#   ar (Semitic, RTL) | hi (Indo-Aryan, SOV) | yue (Sinitic, low-resource) | es (Romance)
#   ko (Koreanic, agglutinative, SOV) | de (Germanic, closest to English)
SELECTED_PAIRS = [("en", "ar"), ("en", "hi"), ("en", "yue"),
                  ("en", "es"), ("en", "ko"), ("en", "de")]
SINGLE_TURN_ONLY = True    # multi-turn dialogues mix speakers; start with single-turn
WINDOW = 64                # N: context window length (sub-words)
MODEL_NAME = "xlm-roberta-base"

LANG_CODE = {"English": "en", "Arabic": "ar", "Hindi": "hi", "Cantonese": "yue", "Mandarin": "zh",
             "Japanese": "ja", "Korean": "ko", "Russian": "ru",
             "Spanish": "es", "French": "fr", "German": "de", "Italian": "it"}

def get_pair(ex):
    """Return ("en", X) from first_language / second_language."""
    langs = {LANG_CODE.get(ex["first_language"]), LANG_CODE.get(ex["second_language"])}
    other = (langs - {"en"}) or {None}
    return ("en", other.pop())

def get_text(ex):
    """data_generation_result may be a plain string or a stringified list of turns."""
    raw = ex[TEXT_COL]
    if isinstance(raw, str) and raw.strip().startswith("["):
        try:
            raw = ast.literal_eval(raw)
        except (ValueError, SyntaxError):
            pass
    return " ".join(map(str, raw)) if isinstance(raw, list) else str(raw)

def keep(ex):
    if SINGLE_TURN_ONLY and ex.get("conversation_type") != "single_turn":
        return False
    return get_pair(ex) in set(SELECTED_PAIRS)

## 3. Word-level LID and predictive label generation
Text is pre-tokenized into words (each Han character counts as one token, since Chinese has no spaces).
Language-neutral tokens (punctuation, digits) inherit the previous token's language.

- **Script-based LID** (ar, hi, yue, ko): the Unicode script of a word decides its language.
- **Frequency-based LID** (es, de): compare the word's Zipf frequency in English vs. the other language (`wordfreq`).
  If both scores are low (unknown word) or too close (`die`, `in`, `no`), the word inherits the previous word's language.

In [25]:
import re, unicodedata

SCRIPT_OF_LANG = {
    "en": "LATIN", "ar": "ARABIC", "hi": "DEVANAGARI", "yue": "HAN", "zh": "HAN",
    "ja": "JAPANESE", "ko": "HANGUL", "ru": "CYRILLIC",
    "es": "LATIN", "fr": "LATIN", "de": "LATIN", "it": "LATIN",
}

def char_script(ch):
    try:
        name = unicodedata.name(ch)
    except ValueError:
        return None
    for key in ["LATIN", "ARABIC", "DEVANAGARI", "CYRILLIC", "HANGUL"]:
        if name.startswith(key):
            return key
    if name.startswith("CJK"):
        return "HAN"
    if name.startswith(("HIRAGANA", "KATAKANA")):
        return "JAPANESE"
    return None

TOKEN_RE = re.compile(r"[\u4e00-\u9fff\u3400-\u4dbf]|[^\s\u4e00-\u9fff\u3400-\u4dbf]+")

def pre_tokenize(text):
    return TOKEN_RE.findall(text)

def word_lid(word, pair, latin_lid_fn=None):
    scripts = [char_script(c) for c in word if c.isalpha()]
    scripts = [s for s in scripts if s]
    if not scripts:
        return None
    script = max(set(scripts), key=scripts.count)
    l1, l2 = pair
    s1, s2 = SCRIPT_OF_LANG[l1], SCRIPT_OF_LANG[l2]
    if s1 != s2:
        if script == "HAN" and "JAPANESE" in (s1, s2):
            script = "JAPANESE"          # kanji inside a Japanese-English pair
        if script == s1: return l1
        if script == s2: return l2
        return None
    if latin_lid_fn is None:
        raise ValueError(f"{pair} share a script; provide latin_lid_fn")
    return latin_lid_fn(word, pair)

def fill_neutral(lids):
    out, prev = [], None
    for l in lids:
        if l is None: l = prev
        out.append(l); prev = l
    first = next((l for l in out if l is not None), None)
    return [l if l is not None else first for l in out]

def dur_bin(n):
    return 0 if n <= 2 else (1 if n <= 6 else 2)

def make_labels(lids):
    T = len(lids)
    run_len = [0] * T
    for i in range(T - 1, -1, -1):
        run_len[i] = 1 + (run_len[i + 1] if i + 1 < T and lids[i + 1] == lids[i] else 0)
    y_sw, y_dur = [], []
    for t in range(T):
        if t == T - 1:
            y_sw.append(-100); y_dur.append(-1); continue
        sw = int(lids[t + 1] != lids[t])
        y_sw.append(sw)
        y_dur.append(dur_bin(run_len[t + 1]) if sw else -1)
    return y_sw, y_dur

# ---------- Frequency-based LID for Latin-script pairs (e.g. en-es, en-de) ----------
WORD_CHARS = re.compile(r"[^\w'-]")

def make_latin_lid(freq_fn, margin=1.0, min_zipf=1.5):
    """freq_fn(word, lang) -> Zipf frequency (wordfreq.zipf_frequency).
    Returns None when a word is unknown in both languages or the scores are too
    close (e.g. 'die', 'no', 'in'); fill_neutral() then uses the previous word's language."""
    def latin_lid(word, pair):
        w = WORD_CHARS.sub("", word)
        if not w or w.isdigit():
            return None
        l1, l2 = pair
        f1, f2 = freq_fn(w, l1), freq_fn(w, l2)
        if max(f1, f2) < min_zipf or abs(f1 - f2) < margin:
            return None
        return l1 if f1 > f2 else l2
    return latin_lid

In [26]:
LATIN_LID = make_latin_lid(zipf_frequency, margin=1.0, min_zipf=1.5)

### Sanity check on toy sentences

In [27]:
words = pre_tokenize("我今日好 busy, 因為 I have three meetings 啊")
lids = fill_neutral([word_lid(w, ("en", "yue")) for w in words])
y_sw, y_dur = make_labels(lids)
pd.DataFrame({"token": words, "LID": lids, "y_sw (t+1)": y_sw, "y_dur": y_dur})

,token,LID,y_sw (t+1),y_dur
0,我,yue,0,-1
1,今,yue,0,-1
2,日,yue,0,-1
3,好,yue,1,0
4,"busy,",en,1,0
5,因,yue,0,-1
6,為,yue,1,1
7,I,en,0,-1
8,have,en,0,-1
9,three,en,0,-1


In [28]:
def show(sentence, pair):
    w = pre_tokenize(sentence)
    raw = [word_lid(x, pair, LATIN_LID) for x in w]
    l = fill_neutral(raw)
    sw, dur = make_labels(l)
    return pd.DataFrame({"token": w, "raw LID": raw, "LID": l, "y_sw": sw, "y_dur": dur})

show("In Zukunft wird er die neuesten Technologien nutzen, um more innovative solutions zu finden.", ("en", "de"))

,token,raw LID,LID,y_sw,y_dur
0,In,NaN,de,0,-1
1,Zukunft,de,de,0,-1
2,wird,de,de,0,-1
3,er,de,de,0,-1
4,die,de,de,0,-1
5,neuesten,de,de,0,-1
6,Technologien,de,de,0,-1
7,"nutzen,",de,de,0,-1
8,um,de,de,1,1
9,more,en,en,0,-1


In [29]:
show("Mañana tengo que ir al office porque tenemos un meeting muy importante.", ("en", "es"))

,token,raw LID,LID,y_sw,y_dur
0,Mañana,es,es,0,-1
1,tengo,es,es,0,-1
2,que,es,es,0,-1
3,ir,es,es,0,-1
4,al,es,es,1,0
5,office,en,en,1,1
6,porque,es,es,0,-1
7,tenemos,es,es,0,-1
8,un,es,es,1,0
9,meeting,en,en,1,0


### Data checks
1. **Hindi script**: if most Hindi rows are romanized (Hinglish in Latin script), script-based LID will not work for them.
2. **Latin-pair review**: export a sample of es/de sentences with their word-level LIDs for manual checking, and report the error rate.

In [30]:
def script_share(texts, script="DEVANAGARI"):
    letters = [char_script(c) for t in texts for c in t if c.isalpha()]
    letters = [x for x in letters if x]
    return round(100 * sum(x == script for x in letters) / max(len(letters), 1), 1)

hi_rows = [get_text(ex) for ex in ds[split].select(range(min(20000, len(ds[split]))))
           if get_pair(ex) == ("en", "hi")][:500]
print(f"Hindi rows checked: {len(hi_rows)}, Devanagari share of letters: {script_share(hi_rows)}%")

Hindi rows checked: 0, Devanagari share of letters: 0.0%


In [31]:
import random
random.seed(0)
review = []
sample = ds[split].select(range(min(20000, len(ds[split]))))
for pair in [("en", "es"), ("en", "de")]:
    rows = [ex for ex in sample if get_pair(ex) == pair]
    for ex in random.sample(rows, min(30, len(rows))):
        words = pre_tokenize(get_text(ex))
        lids = fill_neutral([word_lid(w, pair, LATIN_LID) for w in words])
        review.append({"pair": "-".join(pair),
                       "labeled": " ".join(f"{w}/{l}" for w, l in zip(words, lids)),
                       "correct? (fill in)": ""})
pd.DataFrame(review).to_csv("lid_review.csv", index=False)
pd.DataFrame(review).head()

""


## 4. Sub-word alignment
Each word's language ID is copied to all its sub-words. The word-level labels are placed on the **last sub-word** of each word,
because that is the position from which the model predicts the next word. Other sub-words are ignored (`-100` / `-1`).

In [32]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def align_to_subwords(words, lids, y_sw, y_dur, max_len=512):
    enc = tokenizer(words, is_split_into_words=True, add_special_tokens=False,
                    truncation=True, max_length=max_len)
    wids, ids = enc.word_ids(), enc["input_ids"]
    sw, dur, lid = [-100] * len(ids), [-1] * len(ids), [None] * len(ids)
    for i, w in enumerate(wids):
        if w is None:
            continue
        lid[i] = lids[w]
        nxt = wids[i + 1] if i + 1 < len(wids) else None
        # last sub-word of a word that was not cut off by truncation
        is_last = (nxt is not None and nxt != w) or (nxt is None and w == len(words) - 1)
        if is_last:
            sw[i], dur[i] = y_sw[w], y_dur[w]
    return ids, sw, dur, lid

ids, sw, dur, lid = align_to_subwords(words, lids, y_sw, y_dur)
pd.DataFrame({"subword": tokenizer.convert_ids_to_tokens(ids), "LID": lid, "y_sw": sw, "y_dur": dur})

,subword,LID,y_sw,y_dur
0,▁我,yue,0,-1
1,▁今,yue,0,-1
2,▁,yue,-100,-1
3,日,yue,0,-1
4,▁,yue,-100,-1
5,好,yue,1,0
6,▁busy,en,-100,-1
7,",",en,1,0
8,▁,yue,-100,-1
9,因,yue,0,-1


## 5. Streaming DataLoader
For every labeled position `t`, the dataset yields the prefix window `[x_{t-N+1}, …, x_t]` and the labels for `x_{t+1}`.
The model only ever sees the past. Windows are **left-padded** so the prediction position is always the last one.

In [33]:
def process_example(ex, latin_lid_fn=LATIN_LID):
    pair = get_pair(ex)
    words = pre_tokenize(get_text(ex))
    if len(words) < 2:
        return None
    lids = fill_neutral([word_lid(w, pair, latin_lid_fn) for w in words])
    y_sw, y_dur = make_labels(lids)
    return align_to_subwords(words, lids, y_sw, y_dur)

class StreamingCSDataset(IterableDataset):
    def __init__(self, hf_split, window=WINDOW, pairs=SELECTED_PAIRS, latin_lid_fn=LATIN_LID):
        self.data, self.window = hf_split, window
        self.pairs, self.latin_lid_fn = set(pairs), latin_lid_fn

    def __iter__(self):
        info = get_worker_info()
        wid, nw = (info.id, info.num_workers) if info else (0, 1)
        for i, ex in enumerate(self.data):
            if i % nw != wid or not keep(ex):
                continue
            out = process_example(ex, self.latin_lid_fn)
            if out is None:
                continue
            ids, sw, dur, _ = out
            for t in range(len(ids)):
                if sw[t] == -100:
                    continue
                start = max(0, t - self.window + 1)
                yield {"input_ids": ids[start:t + 1], "y_sw": sw[t], "y_dur": dur[t]}

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    ids = [[pad] * (L - len(b["input_ids"])) + b["input_ids"] for b in batch]
    mask = [[0] * (L - len(b["input_ids"])) + [1] * len(b["input_ids"]) for b in batch]
    return {"input_ids": torch.tensor(ids), "attention_mask": torch.tensor(mask),
            "y_sw": torch.tensor([b["y_sw"] for b in batch]),
            "y_dur": torch.tensor([b["y_dur"] for b in batch])}

train_ds = StreamingCSDataset(ds[split])
loader = DataLoader(train_ds, batch_size=32, collate_fn=collate)
batch = next(iter(loader))
{k: v.shape for k, v in batch.items()}

{'input_ids': torch.Size([32, 37]),
 'attention_mask': torch.Size([32, 37]),
 'y_sw': torch.Size([32]),
 'y_dur': torch.Size([32])}

## 6. Label statistics (for the slides)
Switch events are rare, so report the class balance of `y_sw` and the distribution of `y_dur` bins per language pair.

In [34]:
stats = {p: {"sw": Counter(), "dur": Counter()} for p in SELECTED_PAIRS}
for ex in ds[split].select(range(min(5000, len(ds[split])))):
    if not keep(ex):
        continue
    p = get_pair(ex)
    out = process_example(ex)
    if out is None:
        continue
    _, sw, dur, _ = out
    stats[p]["sw"].update(s for s in sw if s != -100)
    stats[p]["dur"].update(d for d in dur if d != -1)

rows = []
for p, s in stats.items():
    n = sum(s["sw"].values()) or 1
    rows.append({"pair": "-".join(p), "positions": n, "switch %": round(100 * s["sw"][1] / n, 2),
                 "Small": s["dur"][0], "Medium": s["dur"][1], "Large": s["dur"][2]})
pd.DataFrame(rows)

,pair,positions,switch %,Small,Medium,Large
0,en-ar,80268,12.8,3374,2694,4207
1,en-hi,1,0.0,0,0,0
2,en-yue,1,0.0,0,0,0
3,en-es,1,0.0,0,0,0
4,en-ko,1,0.0,0,0,0
5,en-de,1,0.0,0,0,0
